# CCK Fair-Metric Sweep
Runs three retrieval experiments and writes three JSON files to `/kaggle/working/`.
**Download those three files after the run and commit them to `benchmark/` in the repo.**

Outputs:
- `SWEEP_bge_base_full.json` — bge-base-en-v1.5 at full corpus, N=240
- `SWEEP_threshold_0.65.json` — bge-small at full corpus, m3 threshold=0.65, N=240
- `SWEEP_threshold_0.80.json` — bge-small at full corpus, m3 threshold=0.80, N=240

Expected runtime: ~20 min on Kaggle 16 GB CPU.

In [ ]:
!pip install fastembed --quiet
!pip install transformers --quiet
print('deps installed')

In [ ]:
import subprocess
for name, url in [
    ('requests', 'https://github.com/psf/requests.git'),
    ('click',    'https://github.com/pallets/click.git'),
    ('black',    'https://github.com/psf/black.git'),
]:
    r = subprocess.run(
        ['git', 'clone', '--depth=200', url, f'/tmp/{name}'],
        capture_output=True, text=True
    )
    print(f'{name}: {"ok" if r.returncode == 0 else r.stderr.strip()}')

In [ ]:
import glob, os
for name in ['requests', 'click', 'black']:
    py_files = glob.glob(f'/tmp/{name}/**/*.py', recursive=True)
    print(f'{name}: {len(py_files)} .py files')

In [ ]:
# ── All core logic inlined -- no external repo needed ──────────────────────────
from __future__ import annotations
import os, re, json, math, random, statistics, ast, glob
from collections import Counter

# ── BM25 ──────────────────────────────────────────────────────────────────────
_WORD = re.compile(r'[A-Za-z0-9_]+')
_STOP = set('the a an of to in on for and or is are was were be by with vs at as '
            'it its this that what which who whose how many each'.split())

def _tokenize(s):
    return [w.lower() for w in _WORD.findall(s) if w.lower() not in _STOP]

def chunk(text, size=400, overlap=50):
    words = text.split()
    out = []
    i = 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        i += size - overlap
    return out

class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.k1 = k1; self.b = b
        self.N = len(chunks)
        toks = [_tokenize(c) for c in chunks]
        self.avgdl = sum(len(t) for t in toks) / max(self.N, 1)
        self.tf = [Counter(t) for t in toks]
        df = Counter()
        for t in toks:
            for w in set(t): df[w] += 1
        self.idf = {w: math.log((self.N - df[w] + 0.5)/(df[w] + 0.5) + 1) for w in df}
    def score(self, query, idx):
        dl = sum(self.tf[idx].values())
        s = 0.0
        for w in _tokenize(query):
            if w not in self.idf: continue
            tf = self.tf[idx].get(w, 0)
            s += self.idf[w] * (tf*(self.k1+1)) / (tf + self.k1*(1 - self.b + self.b*dl/self.avgdl))
        return s
    def topk(self, query, k):
        scored = sorted(range(self.N), key=lambda i: self.score(query, i), reverse=True)
        return [self.chunks[i] for i in scored[:k]]

# ── Semantic retriever ────────────────────────────────────────────────────────
from fastembed import TextEmbedding

def _cos(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

class SemanticRetriever:
    def __init__(self, chunks, model_name='BAAI/bge-small-en-v1.5', max_chunks=100000, batch=64):
        self.capped = len(chunks) > max_chunks
        if self.capped:
            step = len(chunks) / max_chunks
            chunks = [chunks[int(i*step)] for i in range(max_chunks)]
        self.chunks = chunks
        self.n_indexed = len(chunks)
        self._model = TextEmbedding(model_name)
        emb = []
        for i in range(0, len(chunks), batch):
            emb.extend(self._model.embed(chunks[i:i+batch]))
        self._emb = emb
    def topk(self, query, k):
        if not self._emb: return []
        qv = list(self._model.embed([query]))[0]
        scored = sorted(range(len(self.chunks)), key=lambda i: _cos(qv, self._emb[i]), reverse=True)
        return [self.chunks[i] for i in scored[:k]]

# ── Corpus loader + question miner ───────────────────────────────────────────
def load_repo_text(path):
    parts = []
    for f in glob.glob(os.path.join(path, '**', '*.py'), recursive=True):
        try:
            if os.path.getsize(f) < 200_000:
                parts.append(open(f, encoding='utf-8', errors='ignore').read())
        except OSError:
            pass
    return '\n'.join(parts)

def mine_conceptual(path, n):
    qs = []
    for f in glob.glob(os.path.join(path, '**', '*.py'), recursive=True):
        try:
            if os.path.getsize(f) > 200_000: continue
            tree = ast.parse(open(f, encoding='utf-8', errors='ignore').read())
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                doc = ast.get_docstring(node)
                if not doc: continue
                name = node.name
                if len(name) < 4 or name.startswith('_'): continue
                first = re.split(r'(?<=[.!?])\s', doc.strip())[0].strip()
                if len(first) < 25 or len(first) > 200: continue
                if name.lower() in first.lower(): continue
                qs.append({'q': f'Which function or class {first[0].lower()+first[1:]}', 'fact': name})
    random.Random(0).shuffle(qs)
    seen = set(); out = []
    for it in qs:
        if it['fact'] in seen: continue
        seen.add(it['fact']); out.append(it)
        if len(out) >= n: break
    return out

def bootstrap_ci(hits, nb=1000):
    if not hits: return (None, None, None)
    rng = random.Random(0); N = len(hits); ms = []
    for _ in range(nb):
        ms.append(sum(hits[rng.randrange(N)] for _ in range(N)) / N)
    ms.sort()
    return (round(statistics.mean(hits), 4),
            round(ms[int(0.025*nb)], 4),
            round(ms[int(0.975*nb)], 4))

# ── Metrics ───────────────────────────────────────────────────────────────────
def m1_substring(chunks_list, fact):
    return fact.lower() in '\n'.join(chunks_list).lower()

_defre_cache = {}
def m2_defline(chunks_list, fact):
    if fact not in _defre_cache:
        _defre_cache[fact] = re.compile(r'(?:def|class)\s+' + re.escape(fact) + r'\b')
    return any(_defre_cache[fact].search(c) for c in chunks_list)

print('Core logic loaded.')

In [ ]:
# ── Main sweep function ───────────────────────────────────────────────────────
def run_sweep(repo_paths, n_per_repo, k, bootstrap, max_chunks,
              embed_model, sem_threshold, out_path):
    print(f'\n{"="*60}')
    print(f'model={embed_model}  thresh={sem_threshold}  max_chunks={max_chunks}')
    print(f'{"="*60}', flush=True)

    corpus   = {n: load_repo_text(p)  for n, p in repo_paths.items()}
    chunks_m = {n: chunk(corpus[n])   for n in corpus}
    bm25_m   = {n: BM25(chunks_m[n]) for n in corpus}

    print(f'Embedding ({embed_model})...', flush=True)
    sem_m = {n: SemanticRetriever(chunks_m[n], model_name=embed_model, max_chunks=max_chunks)
             for n in corpus}
    model_inst = sem_m[list(sem_m)[0]]._model

    Q = []
    for n, p in repo_paths.items():
        for it in mine_conceptual(p, n_per_repo):
            if it['fact'].lower() in corpus[n].lower():
                it['project'] = n
                Q.append(it)
    print(f'N={len(Q)} questions across {list(repo_paths.keys())}', flush=True)

    facts    = [q['fact'] for q in Q]
    fact_emb = dict(zip(facts, model_inst.embed([f'definition of {f}' for f in facts])))

    def retrieve(arm, q):
        p = q['project']
        return bm25_m[p].topk(q['q'], k) if arm == 'bm25' else sem_m[p].topk(q['q'], k)

    def score_all(arm):
        h1, h2, h3 = [], [], []
        for q in Q:
            ch   = retrieve(arm, q)
            fact = q['fact']
            s1   = m1_substring(ch, fact)
            s2   = m2_defline(ch, fact)
            if s1:
                s3 = True
            else:
                ce = list(model_inst.embed(ch))
                fe = fact_emb[fact]
                s3 = any(_cos(fe, c) >= sem_threshold for c in ce)
            h1.append(int(s1)); h2.append(int(s2)); h3.append(int(s3))
        return h1, h2, h3

    result = {
        'N': len(Q), 'k': k, 'max_chunks': max_chunks,
        'embed_model': embed_model, 'sem_threshold': sem_threshold,
        'repos': list(repo_paths.keys()),
        'metrics_explained': {
            'm1': 'substring anywhere (favors lexical)',
            'm2': 'def/class defline (stricter, neither favored)',
            'm3': f'substring OR semantic-credit cos>={sem_threshold} (favors semantic)',
        },
        'arms': {}
    }

    print('Scoring...', flush=True)
    for arm in ('bm25', 'semantic'):
        h1, h2, h3 = score_all(arm)
        result['arms'][arm] = {
            'm1_substring':  bootstrap_ci(h1, bootstrap),
            'm2_defline':    bootstrap_ci(h2, bootstrap),
            'm3_sem_credit': bootstrap_ci(h3, bootstrap),
        }

    print(f'\n{"metric":<18}{"bm25 [CI]":>32}{"semantic [CI]":>32}')
    for m in ('m1_substring', 'm2_defline', 'm3_sem_credit'):
        b = result['arms']['bm25'][m]
        s = result['arms']['semantic'][m]
        print(f'{m:<18}{f"{b[0]} [{b[1]},{b[2]}]":>32}{f"{s[0]} [{s[1]},{s[2]}]":>32}')

    with open(out_path, 'w') as f:
        json.dump(result, f, indent=1)
    print(f'wrote {out_path}')
    return result

print('Sweep function ready.')

In [ ]:
# ── Run all three sweeps ──────────────────────────────────────────────────────
REPOS = {
    'requests': '/tmp/requests',
    'click':    '/tmp/click',
    'black':    '/tmp/black',
}
N_PER_REPO = 80       # 80 x 3 = 240 total -- matches the N=240 Kaggle run in CURRENT_STATE.md
K          = 6
BOOTSTRAP  = 1000
MAX_CHUNKS = 100000   # full corpus -- the key variable under test
OUT        = '/kaggle/working'

results = {}

# 1. bge-base at full corpus
results['bge_base'] = run_sweep(
    REPOS, N_PER_REPO, K, BOOTSTRAP, MAX_CHUNKS,
    embed_model   = 'BAAI/bge-base-en-v1.5',
    sem_threshold = 0.72,
    out_path      = f'{OUT}/SWEEP_bge_base_full.json',
)

# 2. bge-small, threshold=0.65
results['thresh_065'] = run_sweep(
    REPOS, N_PER_REPO, K, BOOTSTRAP, MAX_CHUNKS,
    embed_model   = 'BAAI/bge-small-en-v1.5',
    sem_threshold = 0.65,
    out_path      = f'{OUT}/SWEEP_threshold_0.65.json',
)

# 3. bge-small, threshold=0.80
results['thresh_080'] = run_sweep(
    REPOS, N_PER_REPO, K, BOOTSTRAP, MAX_CHUNKS,
    embed_model   = 'BAAI/bge-small-en-v1.5',
    sem_threshold = 0.80,
    out_path      = f'{OUT}/SWEEP_threshold_0.80.json',
)

In [ ]:
# ── Summary table + CI overlap check ─────────────────────────────────────────
print('\n=== FINAL SUMMARY ===')
print('All gaps should show CI_overlap=YES at full corpus.\n')
for label, r in results.items():
    print(f'{label}  N={r["N"]}  model={r["embed_model"]}  thresh={r["sem_threshold"]}')
    for m in ('m1_substring', 'm2_defline', 'm3_sem_credit'):
        b = r['arms']['bm25'][m]
        s = r['arms']['semantic'][m]
        gap     = round(b[0] - s[0], 4)
        overlap = b[1] <= s[2] and s[1] <= b[2]
        flag    = '' if overlap else '  <-- INVESTIGATE'
        print(f'  {m:<22} BM25={b[0]}  sem={s[0]}  gap={gap}  CI_overlap={"YES" if overlap else "NO"}{flag}')
    print()

print('Download SWEEP_*.json from Output tab and commit to benchmark/ in the repo.')